# FerrisRes — training smoke

**What this proves:** the CLI training path learns. `train` calls the real f32
loop (forward → shifted cross-entropy → backward → Adam), writes a checkpoint,
and resumes from it. This is a pipeline proof, not a model.

**Prerequisites**

1. A `.deb` built from a branch that contains `feat/train-cli` **and**
   `feat/train-export` (version **0.2.4** or later), attached at
   `/kaggle/input/ferrisres-deb/`. The 0.2.3 package can train but cannot export.
2. GPU T4 ×2 and internet enabled.
3. No secrets in any cell.

**Gate:** `final_loss < initial_loss`, `params_changed=true`, a checkpoint on
disk, a resume that reports the step it restored, an exported `.safetensors`
that `infer` loads, and at least one generated token.

The corpus in the next cell is synthetic and CC0. It exists so the smoke is
licence-clean and deterministic; it is not release training data.


In [ ]:
from pathlib import Path
import re, subprocess, time

!nvidia-smi -L

DEB_DIR = Path('/kaggle/input/ferrisres-deb')
debs = sorted(DEB_DIR.glob('ferrisres_*_amd64.deb'))
assert debs, 'attach the .deb built from the train-cli branch'
!dpkg -i {debs[-1]}

BIN = subprocess.run(['bash', '-lc', 'command -v ferrisres'],
                     capture_output=True, text=True).stdout.strip()
assert BIN, 'ferrisres is not on PATH after install'
print('binary:', BIN)
print(subprocess.run([BIN, 'devices'], capture_output=True, text=True).stdout)


In [ ]:
# Synthetic, deterministic, CC0. A small repeating vocabulary plus a period-4
# answer, so the byte tokenizer has something learnable without a corpus licence.
words = ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
lines = []
for i in range(100):
    lines.append(' '.join(words[(i + k) % len(words)] for k in range(len(words))))
    lines.append('the answer is %d .' % (i % 4))
text = '\n'.join(lines) + '\n'

CORPUS = Path('/kaggle/working/smoke_corpus.txt')
CORPUS.write_text(text)
print('corpus bytes:', len(text))


In [ ]:
OUT = Path('/kaggle/working/smoke.ckpt')
cmd = [BIN, 'train',
       '--data', str(CORPUS), '--data-format', 'text', '--chat-template', 'gemma',
       '--hidden-dim', '64', '--num-blocks', '2', '--block-size', '4',
       '--epochs', '3', '--batch-size', '4', '--learning-rate', '3e-3',
       '--seq-len', '128', '--vocab', '259', '--seed', '42',
       '--checkpoint-every', '20', '--output', str(OUT), '--log-every', '10']

t0 = time.perf_counter()
proc = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.perf_counter() - t0
log = proc.stdout + proc.stderr
print(log[-4000:])
assert proc.returncode == 0, 'train failed'

def grab(pattern):
    m = re.search(pattern, log)
    assert m, f'missing {pattern} in output'
    return float(m.group(1))

initial = grab(r'initial_loss=([0-9.]+)')
final = grab(r'final_loss=([0-9.]+)')
assert final < initial, f'loss did not fall: {initial} -> {final}'
assert 'params_changed=true' in log, 'no parameter changed'
assert OUT.exists(), 'checkpoint missing'
print(f'initial={initial:.6f} final={final:.6f} elapsed={elapsed:.1f}s')
print(f'steps/s={1.0/max(elapsed,1e-9):.3f} of the whole run (report only)')


In [ ]:
resumed = Path('/kaggle/working/smoke_resumed.ckpt')
cmd2 = cmd[:]
cmd2[cmd2.index('--output') + 1] = str(resumed)
cmd2[cmd2.index('--epochs') + 1] = '1'
cmd2 += ['--resume', str(OUT)]

proc2 = subprocess.run(cmd2, capture_output=True, text=True)
log2 = proc2.stdout + proc2.stderr
print(log2[-3000:])
assert proc2.returncode == 0, 'resume failed'
m = re.search(r'resumed_from=(\d+)', log2)
assert m and int(m.group(1)) > 0, 'resume did not report a restored step'
assert 'final_loss=' in log2, 'resumed run produced no final loss'
assert resumed.exists(), 'resumed checkpoint missing'
print('resumed_from:', m.group(1), '->', resumed)


In [ ]:
import json

EXPORT = Path('/kaggle/working/smoke')
proc3 = subprocess.run(
    [BIN, 'export', '--checkpoint', str(OUT), '--output', str(EXPORT)],
    capture_output=True, text=True,
)
log3 = proc3.stdout + proc3.stderr
print(log3[-3000:])
assert proc3.returncode == 0, 'export failed'

SFT = Path(str(EXPORT) + '.safetensors')
CFG = Path(str(EXPORT) + '.json')
assert SFT.exists(), 'safetensors missing'
assert CFG.exists(), 'config json missing'
summary = json.loads(CFG.read_text())
print('exported tensors:', summary.get('tensor_count'), 'params:', summary.get('parameter_count'))
assert summary.get('tensor_count', 0) > 0

proc4 = subprocess.run(
    [BIN, 'infer', '--model-path', str(SFT), '--model-format', 'safetensors',
     '--prompt', 'the answer is', '--max-tokens', '8', '--temperature', '0'],
    capture_output=True, text=True,
)
log4 = proc4.stdout + proc4.stderr
print(log4[-2000:])
assert proc4.returncode == 0, 'infer failed'
assert 'generated_tokens=' in log4, 'infer produced no token count'
print('export + infer ok')


## What to record

- the adapter name from `ferrisres devices` (a software adapter invalidates the
  timing, not the correctness);
- `initial_loss` and `final_loss`;
- wall clock and the derived steps/s;
- checkpoint file sizes, and that `--resume` reported the step it restored.

## What this does **not** prove

- learning on a real corpus at scale (the corpus here is synthetic);
- the BPE/32k tokenizer (this uses the 259-token byte tokenizer);
- the block+LoRA path (it has the separate 4.89 gradient defect and is refused);
- anything about two GPUs (the multi-GPU launcher is not wired into training yet);
- f32 fidelity through the inference loader: the CPU loader quantises loaded
  weights to ternary, so the export/infer step proves plumbing, not numerical
  equality with the trainer;
- GGUF or ternary export (not implemented);
- GPU training: this palindrome is the CPU f32 loop until the wgpu trainer lands.
